In [28]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
import os
from tempfile import NamedTemporaryFile, TemporaryDirectory # Creating temporary Files/Dirs
from subprocess import run, PIPE
import sys
 
import dask # Distributed data libary
from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
import xarray as xr # Libary to work with labeled n-dimensional data and dask

In [29]:
# Set some user specific variables
scratch_dir = Path('/scratch') / getuser()[0] / getuser() # Define the users scratch dir
# Create a temp directory where the output of distributed cluster will be written to, after this notebook
# is closed the temp directory will be closed
dask_tmp_dir = TemporaryDirectory(dir=scratch_dir, prefix='PostProc')
cluster = SLURMCluster(memory='400GiB',
                       cores=72,
                       project='mh0731',
                       walltime='1:00:00',
                       queue='compute',
                       name='PostProc',
                   #    scheduler_options={'dashboard_address': ':12435'},
                       local_directory=dask_tmp_dir.name,
                       job_extra=[f'-J PostProc', 
                                  f'-D {dask_tmp_dir.name}',
                                  f'--begin=now',
                                  f'--output={dask_tmp_dir.name}/LOG_cluster.%j.o',
                                  f'--output={dask_tmp_dir.name}/LOG_cluster.%j.o'
                                 ],
                       interface='ib0')
cluster.scale(jobs=1)
dask_client = Client(cluster)

/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/dask_jobqueue/slurm.py:55: FutureWarning: project has been renamed to account as this kwarg was used wit -A option. You are still using it (please also check config files). If you did not set account yet, project will be respected for now, but it will be removed in a future release. If you already set account, project is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/

In [30]:
from getpass import getuser # Libaray to copy things
from tempfile import NamedTemporaryFile, TemporaryDirectory 

# calculation
#import metpy.calc as mpcalc

# scipy
from scipy import stats
from scipy.ndimage import measurements
from scipy import ndimage
from scipy.optimize import curve_fit

# for plot
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# basic
from pathlib import Path # Object oriented libary to deal with paths
import netCDF4 as nc
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import glob
import os

#metpy
#import metpy.calc as mpcalc
#from metpy.units import units

## Save RH and TD

In [4]:
# ctl
ta_ctl_directory = '/scratch/m/m300948/test_04/3d/ta/6hourly'
qa_ctl_directory = '/scratch/m/m300948/test_04/3d/specif_humidity/6hourly'
rh_ctl_directory = '/scratch/m/m300948/test_04/3d/rh'
td_ctl_directory = '/scratch/m/m300948/test_04/3d/dew_point_temp'

In [5]:
# def
ta_def_directory = '/scratch/m/m300948/def_100/3d/ta/6hourly'
qa_def_directory = '/scratch/m/m300948/def_100/3d/specif_humidity/6hourly'
rh_def_directory = '/scratch/m/m300948/def_100/3d/rh'
td_def_directory = '/scratch/m/m300948/def_100/3d/dew_point_temp'

## 1) Make Td and Ta

In [7]:
qa_files = sorted(glob.glob(os.path.join(qa_ctl_directory, "ctl_daily_hus_plev_2022*.nc")))

def calculate_and_save_rh(qa_file):
    # Derive the corresponding temperature file name
    filename = os.path.basename(qa_file)
    ta_file = os.path.join(ta_ctl_directory, filename.replace("ctl_daily_hus_plev_2022", "ctl_daily_ta_plev_2022"))
    rh_file = os.path.join(rh_ctl_directory, filename.replace("ctl_daily_hus_plev_2022", "ctl_daily_rh_plev_2022"))
    
    if not os.path.exists(ta_file):
        print(f"Temperature file {ta_file} not found. Skipping...")
        return
    
    # Open the specific humidity and temperature files
    ds_qa = xr.open_dataset(qa_file)
    ds_ta = xr.open_dataset(ta_file)
    
    print(ds_qa)
    print(ds_ta)
    # Extract variables (assuming standard names, modify if needed)
    specific_humidity = ds_qa["hus"]
    temperature = (ds_ta["ta"] - 273.15) * units.degC
    pressure = ds_qa["plev"]/100 * units.hPa  # Assuming pressure is a coordinate in the dataset
    
    # Compute relative humidity
    #relative_humidity_from_specific_humidity(press_hpa * units.hPa, ta_def_degc , qa_def_mask)
    rh = mpcalc.relative_humidity_from_specific_humidity(pressure, temperature, specific_humidity)
    
    # Add relative humidity to a new dataset
    ds_rh = ds_qa.copy()
    ds_rh["rh"] = rh.metpy.dequantify()  # Keep the same dimensions
    ds_rh["rh"].attrs = {"units": "%", "description": "Relative Humidity"}
    
    # Save the new dataset
    ds_rh.to_netcdf(rh_file)
    print(f"Saved {rh_file}")
    
    # Close datasets
    ds_qa.close()
    ds_ta.close()
    ds_rh.close()

In [ ]:
# Process each file
for qa_file in qa_files:
    calculate_and_save_rh(qa_file)

In [15]:
ta_files = sorted(glob.glob(os.path.join(ta_ctl_directory, "ctl_daily_ta_plev_2022*.nc")))

def calculate_and_save_td(ta_file):
    # Derive the corresponding temperature file name
    filename = os.path.basename(ta_file)
    rh_file = os.path.join(rh_ctl_directory, filename.replace("ctl_daily_ta_plev_2022", "ctl_daily_rh_plev_2022"))
    td_file = os.path.join(td_ctl_directory, filename.replace("ctl_daily_ta_plev_2022", "ctl_daily_td_plev_2022"))
    
    if not os.path.exists(ta_file):
        print(f"Temperature file {ta_file} not found. Skipping...")
        return
    
    # Open the specific humidity and temperature files
    ds_ta = xr.open_dataset(ta_file)
    ds_rh = xr.open_dataset(rh_file)
    
    # Extract variables (assuming standard names, modify if needed)
    temperature = (ds_ta["ta"] - 273.15) * units.degC
    relative_humidity = ds_rh["rh"] * units.dimensionless  # Assuming pressure is a coordinate in the dataset
    
    # Compute relative humidity
    #relative_humidity_from_specific_humidity(press_hpa * units.hPa, ta_def_degc , qa_def_mask)
    td = mpcalc.dewpoint_from_relative_humidity(temperature, relative_humidity)
    
    # Add relative humidity to a new dataset
    ds_td = ds_ta.copy()
    ds_td["td"] = td.metpy.dequantify()  # Keep the same dimensions
    ds_td["td"].attrs = {"units": "degC", "description": "dew point temperature"}
    
    # Save the new dataset
    ds_td.to_netcdf(td_file)
    print(f"Saved {td_file}")
    
    # Close datasets
    ds_ta.close()
    ds_rh.close()
    ds_td.close()

In [ ]:
# Process each file
for ta_file in ta_files:
    calculate_and_save_td(ta_file)

## 2) Select the heavy rainy days inside the Amazon region

In [31]:
# For selecting violent rains, I need precipitation data
data_path_ctl = '/scratch/m/m300948/test_04/pr/ctl_hourly_pr_2022.nc'
pr_ctl = xr.open_mfdataset(data_path_ctl, parallel=True)['pr']
data_path_def = '/scratch/m/m300948/def_100/pr/def_daily_pr_202212*'
pr_def = xr.open_mfdataset(data_path_def, parallel=True, chunks={'time': 1, 'lat': -1, 'lon': -1})['pr']

In [32]:
pr_ctl = pr_ctl.isel(time=pr_ctl.time.dt.month == 12)

_load qa_

In [33]:
qa_ctl = xr.open_mfdataset('/scratch/m/m300948/test_04/3d/specif_humidity/6hourly/ctl_daily_hus_plev_202212*', chunks={'time': 1, 'plev': -1, 'lat': -1, 'lon': -1})['hus']

In [34]:
qa_def = xr.open_mfdataset('/scratch/m/m300948/def_100/3d/specif_humidity/6hourly/def_daily_hus_plev_202212*', chunks={'time': 1, 'plev': -1, 'lat': -1, 'lon': -1})['hus']

_load td_

In [35]:
data_path_td_ctl = '/scratch/m/m300948/test_04/3d/dew_point_temp/'
ta_ctl = xr.open_mfdataset(data_path_td_ctl+'ctl_daily_td_plev_202212*', chunks={'time': 1, 'plev': -1, 'lat': -1, 'lon': -1})['ta']
td_ctl = xr.open_mfdataset(data_path_td_ctl+'ctl_daily_td_plev_202212*', chunks={'time': 1, 'plev': -1, 'lat': -1, 'lon': -1})['td']

In [36]:
data_path_td_def = '/scratch/m/m300948/def_100/3d/dew_point_temp/'
ta_def = xr.open_mfdataset(data_path_td_def+'def_daily_td_plev_202212*', chunks={'time': 1, 'plev': -1, 'lat': -1, 'lon': -1})['ta']
td_def = xr.open_mfdataset(data_path_td_def+'def_daily_td_plev_202212*', chunks={'time': 1, 'plev': -1, 'lat': -1, 'lon': -1})['td']

### 2-1) Mask the Amazon region

In [37]:
dset_bd = xr.open_dataset('/work/mh0731/m300948/AMDEF/REGRID_BC/masking_files/AMAZON_Biome.nc')
AMZ_BD = dset_bd.AMAZON_BIOMES

In [38]:
ABinterp_biome_ctl = AMZ_BD.interp(latitude=pr_def.lat, longitude=pr_def.lon)

In [39]:
pr_ctl_mask = pr_ctl.where(ABinterp_biome_ctl == 0, np.nan)
pr_def_mask = pr_def.where(ABinterp_biome_ctl == 0, np.nan)

In [40]:
# select the wa when intense precipitation
ta_ctl_mask = ta_ctl.where(ABinterp_biome_ctl == 0, np.nan)
td_ctl_mask = td_ctl.where(ABinterp_biome_ctl == 0, np.nan)
ta_def_mask = ta_def.where(ABinterp_biome_ctl == 0, np.nan)
td_def_mask = td_def.where(ABinterp_biome_ctl == 0, np.nan)

In [41]:
qa_ctl_mask = qa_ctl.where(ABinterp_biome_ctl == 0, np.nan)
qa_def_mask = qa_def.where(ABinterp_biome_ctl == 0, np.nan)

### 2-2) select datasets when intense precipitation (6n)

In [18]:
# select every 6n hour instantenous precipitation (0, 6, 12, 18) / 6n+1 hour precipitation (1, 7, 13, 19)
pr_ctl_7n = pr_ctl_mask.loc[pr_ctl_mask.time.dt.hour.isin([0, 6, 12, 18]) & (pr_ctl_mask.time.dt.minute == 0)] *3600
pr_def_7n = pr_def_mask.loc[pr_def_mask.time.dt.hour.isin([0, 6, 12, 18]) & (pr_def_mask.time.dt.minute == 0)] *3600

In [19]:
## intense precipitation for each grid point
intense_pr_mask_ctl_7n = pr_ctl_7n.where(pr_ctl_7n > 50)
intense_pr_mask_def_7n = pr_def_7n.where(pr_def_7n > 50)
extreme_pr_mask_ctl = pr_ctl_7n > 50 # true/false 
extreme_pr_mask_def = pr_def_7n > 50 

In [20]:
# select the wa when intense precipitation
selected_ta_ctl= ta_ctl_mask.where(extreme_pr_mask_ctl)
selected_td_ctl= td_ctl_mask.where(extreme_pr_mask_ctl)
selected_ta_def= ta_def_mask.where(extreme_pr_mask_def)
selected_td_def= td_def_mask.where(extreme_pr_mask_def)

In [21]:
# select the wa when intense precipitation
selected_qa_ctl= qa_ctl_mask.where(extreme_pr_mask_ctl)
selected_qa_def= qa_def_mask.where(extreme_pr_mask_def)

In [22]:
%time
ta = selected_ta_def.compute()

CPU times: user 5 μs, sys: 1 μs, total: 6 μs
Wall time: 10 μs


In [23]:
qa = selected_qa_ctl.compute()

KeyboardInterrupt: 

In [ ]:
qa_def = selected_qa_def.compute()

In [49]:
ta.to_netcdf('/scratch/m/m300948/def_100/pre_cape_cin_calculation/selected_ta_def.nc')

In [41]:
qa.to_netcdf('/scratch/m/m300948/test_04/pre_cape_cin_calculation/selected_qa_ctl.nc')

In [43]:
qa_def.to_netcdf('/scratch/m/m300948/def_100/pre_cape_cin_calculation/selected_qa_def.nc')

In [15]:
%time
td = selected_td_def.compute()

CPU times: user 1e+03 ns, sys: 0 ns, total: 1e+03 ns
Wall time: 5.48 µs


In [16]:
td.to_netcdf('/scratch/m/m300948/def_100/pre_cape_cin_calculation/selected_td_def.nc')

In [ ]:
d_file = '/scratch/m/m300948/def_100/pre_cape_cin_calculation/selected_td_def.nc'
ds_td = selected_td_def.copy()
ds_td["td"] = selected_td_def.metpy.dequantify()  # Keep the same dimensions
ds_td["td"].attrs = {"units": "kelvin", "description": "selected temperature"}

# Save the new datdset
ds_td.to_netcdf(d_file)
print(f"Saved {d_file}")

### 2-3) Check when they have common time

## 3) select datasets (ta, qa) one hour before the intense precipitation (6n+1)

In [ ]:
#ta_ctl_mask
#td_ctl_mask
#qa_ctl_mask

### 3-1) select precipitation (6n+1) with intense precipitation

In [42]:
# time
n = 3
# Correct
ctl_mask = (pr_ctl_mask.time.dt.hour % 6 == n) & (pr_ctl_mask > 50) # true/false
def_mask = (pr_def_mask.time.dt.hour % 6 == n) & (pr_def_mask > 50) # true/false

# Shift the mask by 1 hour to align with 6-hourly data
ctl_mask_shifted = ctl_mask.shift(time=-n) # Shift the mask by 1 hour
def_mask_shifted = def_mask.shift(time=-n) # Shift the mask by 1 hour

# Resample the mask to 6-hourly frequency
ctl_mask_6hourly = ctl_mask_shifted.resample(time='6H').any()
def_mask_6hourly = def_mask_shifted.resample(time='6H').any()

/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/xarray/groupers.py:509: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(
/home/m/m300948/.conda/envs/easy/lib/python3.12/site-packages/xarray/groupers.py:509: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


In [137]:
# Wrong version
# select every 6n hour instantenous precipitation (0, 6, 12, 18) / 6n+1 hour precipitation (1, 7, 13, 19)
#pr_ctl_7n = pr_ctl_mask.loc[pr_ctl_mask.time.dt.hour.isin([0, 6, 12, 18]) & (pr_ctl_mask.time.dt.minute == 0)] *3600
#pr_def_7n = pr_def_mask.loc[pr_def_mask.time.dt.hour.isin([0, 6, 12, 18]) & (pr_def_mask.time.dt.minute == 0)] *3600

## intense precipitation for each grid point
#intense_pr_mask_ctl_7n = pr_ctl_7n.where(pr_ctl_7n > 50)
#intense_pr_mask_def_7n = pr_def_7n.where(pr_def_7n > 50)

#extreme_pr_mask_ctl = pr_ctl_7n > 50 # true/false 
#extreme_pr_mask_def = pr_def_7n > 50 

#shifted_extreme_pr_mask_ctl = extreme_pr_mask_ctl.shift(time=-1)
#shifted_extreme_pr_mask_def = extreme_pr_mask_def.shift(time=-1)

In [43]:
# Correct
# Select the ta_ctl_mask values where the shifted mask is True
selected_ta_ctl = ta_ctl_mask.where(ctl_mask_6hourly)
selected_td_ctl = td_ctl_mask.where(ctl_mask_6hourly)
selected_qa_ctl = qa_ctl_mask.where(ctl_mask_6hourly)

In [139]:
# Wrong
# Select the ta_ctl_mask values where the shifted mask is True
#selected_ta_ctl = ta_ctl_mask.where(shifted_extreme_pr_mask_ctl)
#selected_td_ctl = td_ctl_mask.where(shifted_extreme_pr_mask_ctl)
#selected_qa_ctl = qa_ctl_mask.where(shifted_extreme_pr_mask_ctl)

In [44]:
# Correct
# Select the ta_def_mask values where the shifted mask is True
selected_ta_def = ta_def_mask.where(def_mask_6hourly)
selected_td_def = td_def_mask.where(def_mask_6hourly)
selected_qa_def = qa_def_mask.where(def_mask_6hourly)

In [141]:
# wrong
# Select the ta_def_mask values where the shifted mask is True
#selected_ta_def = ta_def_mask.where(shifted_extreme_pr_mask_def)
#selected_td_def = td_def_mask.where(shifted_extreme_pr_mask_def)
#selected_qa_def = qa_def_mask.where(shifted_extreme_pr_mask_def)

In [34]:
selected_td_def.max().values

array(28.46252932)

_ctl_

In [45]:
%time
ta_ctl_6plus1= selected_ta_ctl.compute()

CPU times: user 7 μs, sys: 0 ns, total: 7 μs
Wall time: 12.6 μs


In [46]:
%time
td_ctl_6plus1= selected_td_ctl.compute()

CPU times: user 5 μs, sys: 1 μs, total: 6 μs
Wall time: 10.7 μs


In [47]:
%time
qa_ctl_6plus1= selected_qa_ctl.compute()

CPU times: user 5 μs, sys: 1 μs, total: 6 μs
Wall time: 12.2 μs


_def_

In [48]:
%time
ta_def_6plus1= selected_ta_def.compute()

CPU times: user 5 μs, sys: 1e+03 ns, total: 6 μs
Wall time: 11 μs


In [49]:
%time
td_def_6plus1= selected_td_def.compute()

CPU times: user 4 μs, sys: 2 μs, total: 6 μs
Wall time: 10.5 μs


In [50]:
qa_def_6plus1= selected_qa_def.compute()

## Save the file

In [51]:
suffix = '_12' 
dir_ctl = '/scratch/m/m300948/test_04/pre_cape_cin_calculation/'

In [52]:
ta_ctl_6plus1.to_netcdf(dir_ctl+'selected_ta_ctl_6plus1_3h_correct'+suffix+'.nc')
td_ctl_6plus1.to_netcdf(dir_ctl+'selected_td_ctl_6plus1_3h_correct'+suffix+'.nc')
qa_ctl_6plus1.to_netcdf(dir_ctl+'selected_qa_ctl_6plus1_3h_correct'+suffix+'.nc')

In [53]:
suffix = '_12' 
dir_def = '/scratch/m/m300948/def_100/pre_cape_cin_calculation/'

In [54]:
ta_def_6plus1.to_netcdf(dir_def+'selected_ta_def_6plus1_3h_correct'+suffix+'.nc')
td_def_6plus1.to_netcdf(dir_def+'selected_td_def_6plus1_3h_correct'+suffix+'.nc')
qa_def_6plus1.to_netcdf(dir_def+'selected_qa_def_6plus1_3h_correct'+suffix+'.nc')